# SSIF_V3 模型訓練 Notebook（繁體中文）

流程：training-loader 驗證 → event-disjoint train/validation/calibration/test split → EW10 quick test → EW10–EW40 正式訓練 → checkpoint/fingerprint 稽核。

科學隔離：validation 選最佳 epoch；calibration 選 alert threshold；test 僅在兩者固定後評估。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 同步 repository


In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_ROOT=Path('/content/SSIF_V3')
REPO_URL='https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(cmd,cwd=None,capture=False):
    p=subprocess.run(cmd,cwd=cwd,text=True,capture_output=capture)
    if capture:
        if p.stdout: print(p.stdout,end='')
        if p.stderr: print(p.stderr,end='')
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: "+" ".join(map(str,cmd)))
    return p

os.chdir('/content')
if (REPO_ROOT/'.git').is_dir():
    try:
        run_checked(['git','-C',str(REPO_ROOT),'fetch','--prune','origin'])
        run_checked(['git','-C',str(REPO_ROOT),'reset','--hard','origin/main'])
        run_checked(['git','-C',str(REPO_ROOT),'clean','-fd'])
    except RuntimeError:
        os.chdir('/content'); shutil.rmtree(REPO_ROOT,ignore_errors=True)
        run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])
else:
    shutil.rmtree(REPO_ROOT,ignore_errors=True)
    run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])

REPO_SHA=run_checked(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],capture=True).stdout.strip()
print('Repository commit:',REPO_SHA)
run_checked(['python','-m','pip','install','-q','-r',str(REPO_ROOT/'requirements.txt')])


## 2. 路徑與設定


In [ ]:
from datetime import datetime, timezone
import gc, json, platform, random, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

WORK_ROOT=Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')
TRAIN_DATA_CANDIDATES=[
    Path('/content/drive/MyDrive/00_SSIF/觀測資料'),
    WORK_ROOT/'data'/'training_archive_json',
]
TRAIN_DATA=next((p for p in TRAIN_DATA_CANDIDATES if p.is_dir() and any(p.rglob('*.json'))),TRAIN_DATA_CANDIDATES[0])
PREPARED_DIR=WORK_ROOT/'prepared'/'split_v1'
SPLIT_MANIFEST=PREPARED_DIR/'split_manifest.json'
QUICK_MODEL_DIR=WORK_ROOT/'models'/'quick_EW10'
FULL_MODEL_DIR=WORK_ROOT/'models'/'ssif_v3_seed_20260728'
REPORT_DIR=WORK_ROOT/'reports'/'training_seed_20260728'
EXTERNAL_DATA=WORK_ROOT/'data'/'external_evaluation_json'
EXTERNAL_OUTPUT_DIR=WORK_ROOT/'inference'/'external_seed_20260728'

WINDOWS = [10,15,20,25,30,35,40]
SEED=20260728
LABEL_HORIZON = 120
MIN_VALID=0.80
MIN_PRECISION=0.90
BATCH_SIZE=16
EVAL_BATCH_SIZE=64
WORKERS=2

RUN_DATA_VALIDATION=True
CREATE_SPLIT_IF_MISSING=True
REBUILD_SPLIT = False
AUTO_REBUILD_SPLIT_ON_ARCHIVE_CHANGE=True
RUN_QUICK_TRAIN = True
RUN_FULL_TRAIN = False
RUN_EXTERNAL_EVALUATION = False
OVERWRITE_QUICK_MODEL=True
OVERWRITE_FULL_MODEL = False

for p in [PREPARED_DIR,QUICK_MODEL_DIR.parent,REPORT_DIR,EXTERNAL_OUTPUT_DIR.parent]:
    p.mkdir(parents=True,exist_ok=True)
assert TRAIN_DATA.is_dir(),f'找不到 TRAIN_DATA：{TRAIN_DATA}'
EVENT_FILES=sorted(TRAIN_DATA.rglob('*.json'))
assert EVENT_FILES,f'找不到 JSON：{TRAIN_DATA}'
print('Training archive:',TRAIN_DATA)
print('JSON files:',len(EVENT_FILES))
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. 使用與訓練相同的 loader 驗證 archive

不要使用 `combined_csv_to_ssif_json.py validate` 驗證任意原始 archive；該指令只掃描頂層 `event_*.json`。此處直接使用 `ssif_core.load_station_records()`。


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True

environment={
    'created_at_utc':datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00','Z'),
    'repository_commit':REPO_SHA,
    'training_archive':str(TRAIN_DATA),
    'python':platform.python_version(),
    'torch':torch.__version__,
    'cuda':torch.version.cuda,
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed':SEED,
}
(REPORT_DIR/'environment.json').write_text(json.dumps(environment,ensure_ascii=False,indent=2),encoding='utf-8')

archive_stats=None
if RUN_DATA_VALIDATION:
    sys.path.insert(0,str(REPO_ROOT))
    from ssif_core import load_station_records
    records,archive_stats=load_station_records(
        TRAIN_DATA,
        min_full_valid_fraction=MIN_VALID,
        min_series_length=max(WINDOWS),
        label_horizon=LABEL_HORIZON,
        require_label_horizon=True,
    )
    assert archive_stats['n_files']==len(EVENT_FILES)
    assert archive_stats['n_events']>0
    assert archive_stats['n_records']>0
    assert archive_stats['skipped_files']<archive_stats['n_files']
    report={'status':'ok','loader':'ssif_core.load_station_records','training_archive':str(TRAIN_DATA.resolve()),'stats':archive_stats}
    (REPORT_DIR/'archive_validation.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    keys=['n_files','n_events','n_records','skipped_files','skipped_stations','missing_fraction','label_horizon']
    display(pd.DataFrame([{'metric':k,'value':archive_stats[k]} for k in keys]))
    if archive_stats['skipped_files']:
        print('Warning: skipped_files =',archive_stats['skipped_files'],'；請查看後續 file_audit.csv。')
    del records; gc.collect()
    print('PASS: archive is usable by the training loader')


## 4. 固定 event-disjoint split


In [ ]:
def manifest_matches_archive(m):
    try:
        same_root=Path(str(m.get('data_root',''))).resolve()==TRAIN_DATA.resolve()
    except Exception:
        same_root=str(m.get('data_root',''))==str(TRAIN_DATA)
    same_events=(archive_stats is None or int(m.get('n_events',-1))==int(archive_stats['n_events']))
    return same_root and same_events and list(m.get('windows',[]))==WINDOWS and int(m.get('label_horizon',-1))==LABEL_HORIZON

need_rebuild=REBUILD_SPLIT
if SPLIT_MANIFEST.is_file() and not need_rebuild:
    old=json.loads(SPLIT_MANIFEST.read_text(encoding='utf-8'))
    if not manifest_matches_archive(old):
        if AUTO_REBUILD_SPLIT_ON_ARCHIVE_CHANGE:
            print('Existing split does not match current archive; rebuilding.')
            need_rebuild=True
        else:
            raise RuntimeError('split_manifest.json 不符合目前 TRAIN_DATA；請設定 REBUILD_SPLIT=True。')

if need_rebuild and PREPARED_DIR.exists(): shutil.rmtree(PREPARED_DIR)
PREPARED_DIR.mkdir(parents=True,exist_ok=True)

if not SPLIT_MANIFEST.is_file():
    assert CREATE_SPLIT_IF_MISSING
    run_checked([
        'python','prepare_ssif_dataset.py','audit-split',
        '--data-dir',str(TRAIN_DATA),'--output-dir',str(PREPARED_DIR),
        '--windows',*map(str,WINDOWS),'--label-horizon',str(LABEL_HORIZON),
        '--min-label-valid-fraction',str(MIN_VALID),'--min-window-valid-fraction',str(MIN_VALID),
        '--train-ratio','0.70','--validation-ratio','0.10','--calibration-ratio','0.10','--test-ratio','0.10',
        '--split-candidates','5000','--seed',str(SEED),
    ],cwd=REPO_ROOT)

manifest=json.loads(SPLIT_MANIFEST.read_text(encoding='utf-8'))
assert manifest['validation']['valid']
assert manifest_matches_archive(manifest)
audit=json.loads((PREPARED_DIR/'audit_summary.json').read_text(encoding='utf-8'))
assert audit['n_duplicate_event_ids']==0,'請檢查 duplicate_events.json'
print('data_fingerprint_sha256:',manifest['data_fingerprint_sha256'])
display(pd.DataFrame([{'split':k,'events':len(v)} for k,v in manifest['splits'].items()]))


## 5. Quick／正式訓練


In [ ]:
def train_command(output_dir,windows,epochs):
    cmd=[
        'python','train_ssif_v3.py','train-all',
        '--data-dir',str(TRAIN_DATA),'--split-manifest',str(SPLIT_MANIFEST),
        '--output-dir',str(output_dir),'--windows',*map(str,windows),
        '--label-horizon',str(LABEL_HORIZON),"--cohort",'common',
        '--epochs',str(epochs),'--batch-size',str(BATCH_SIZE),'--eval-batch-size',str(EVAL_BATCH_SIZE),
        '--lr','3e-4','--weight-decay','1e-2','--warmup-ratio','0.10',
        '--min-precision',str(MIN_PRECISION),'--hidden-size','192','--num-layers','4',
        '--num-heads','4','--ff-mult','2','--dropout','0.1','--conv1','96','--conv2','192',
        '--loss-cls','0.45','--loss-alert','0.35','--loss-ordinal','0.15','--loss-consistency','0.05',
        '--seed',str(SEED),"--window-seed-mode",'same','--patience','6','--workers',str(WORKERS),
    ]
    if torch.cuda.is_available(): cmd.append('--amp')
    return cmd

if RUN_QUICK_TRAIN:
    if QUICK_MODEL_DIR.exists() and any(QUICK_MODEL_DIR.iterdir()):
        assert OVERWRITE_QUICK_MODEL
        shutil.rmtree(QUICK_MODEL_DIR)
    run_checked(train_command(QUICK_MODEL_DIR,[10],1),cwd=REPO_ROOT)
    assert (QUICK_MODEL_DIR/'EW10'/'best.pt').is_file()
    quick=json.loads((QUICK_MODEL_DIR/'summary.json').read_text(encoding='utf-8'))[0]
    display(pd.DataFrame([{'window':quick['window'],'best_epoch':quick['best_epoch'],'threshold':quick['threshold'],**quick['test']['alert']}]))
    print('PASS: EW10 quick training')

if RUN_FULL_TRAIN:
    if FULL_MODEL_DIR.exists() and any(FULL_MODEL_DIR.iterdir()):
        assert OVERWRITE_FULL_MODEL,'正式模型已存在；請改 run 名稱或明確允許覆蓋。'
        shutil.rmtree(FULL_MODEL_DIR)
    run_checked(train_command(FULL_MODEL_DIR,WINDOWS,30),cwd=REPO_ROOT)
    assert (FULL_MODEL_DIR/'summary.json').is_file()
    for w in WINDOWS:
        assert (FULL_MODEL_DIR/f'EW{w:02d}'/'best.pt').is_file()
    print('PASS: full training')


## 6. 結果與 checkpoint audit


In [ ]:
def result_table(model_dir):
    p=model_dir/'summary.json'
    if not p.is_file(): return pd.DataFrame()
    rows=[]
    for x in json.loads(p.read_text(encoding='utf-8')):
        a=x['test']['alert']
        rows.append({'window':x['window'],'best_epoch':x['best_epoch'],'threshold':x['threshold'],'precision':a['precision'],'pod':a['pod'],'f1':a['f1'],'fpr':a['fpr'],'n':x['test']['n_samples']})
    return pd.DataFrame(rows).sort_values('window')

results=result_table(FULL_MODEL_DIR)
if not results.empty:
    display(results)
    results.to_csv(REPORT_DIR/'model_performance_by_window.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(9,5))
    for col in ['precision','pod','f1']: plt.plot(results['window'],results[col],marker='o',label=col.upper())
    plt.xlabel('Early window (s)'); plt.ylabel('Score'); plt.ylim(0,1.02); plt.xticks(WINDOWS); plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig(REPORT_DIR/'test_metrics_by_window.png',dpi=180); plt.show()

rows=[]
for w in WINDOWS:
    p=FULL_MODEL_DIR/f'EW{w:02d}'/'best.pt'
    if not p.is_file():
        rows.append({'window':w,'exists':False}); continue
    payload=torch.load(p,map_location='cpu',weights_only=False)
    meta=payload.get('training_metadata',{})
    rows.append({'window':w,'exists':True,'checkpoint_window':payload.get('window'),'best_epoch':meta.get('best_epoch'),'threshold':payload.get('alert_probability_threshold'),'fingerprint_matches':meta.get('data_fingerprint_sha256')==manifest['data_fingerprint_sha256'],'label_horizon':meta.get('label_horizon'),'cohort':meta.get('cohort')})
checkpoint_audit=pd.DataFrame(rows)
display(checkpoint_audit)
if RUN_FULL_TRAIN:
    ok=checkpoint_audit[checkpoint_audit['exists']]
    assert len(ok)==len(WINDOWS) and ok['fingerprint_matches'].all()


## 7. 獨立 archive 評估與 run inventory


In [ ]:
if RUN_EXTERNAL_EVALUATION:
    assert EXTERNAL_DATA.is_dir() and any(EXTERNAL_DATA.rglob('*.json'))
    assert (FULL_MODEL_DIR/'summary.json').is_file()
    if EXTERNAL_OUTPUT_DIR.exists(): shutil.rmtree(EXTERNAL_OUTPUT_DIR)
    run_checked([
        'python','train_ssif_v3.py','evaluate-all',
        '--data-dir',str(EXTERNAL_DATA),'--model-root',str(FULL_MODEL_DIR),
        '--output-dir',str(EXTERNAL_OUTPUT_DIR),'--windows',*map(str,WINDOWS),
        '--label-horizon',str(LABEL_HORIZON),"--cohort",'common',
        '--batch-size','128','--workers',str(WORKERS),
    ],cwd=REPO_ROOT)

run_inventory={
    'created_at_utc':datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00','Z'),
    'repository_commit':REPO_SHA,
    'training_archive':str(TRAIN_DATA.resolve()),
    'data_fingerprint_sha256':manifest['data_fingerprint_sha256'],
    'split_manifest':str(SPLIT_MANIFEST),
    'windows':WINDOWS,
    'label_horizon':LABEL_HORIZON,
    'quick_model_dir':str(QUICK_MODEL_DIR),
    'full_model_dir':str(FULL_MODEL_DIR),
    'external_output_dir':str(EXTERNAL_OUTPUT_DIR),
}
(REPORT_DIR/'run_inventory.json').write_text(json.dumps(run_inventory,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(run_inventory,ensure_ascii=False,indent=2))
